In [6]:
# Imports / global contants

# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import math
import numpy as np
import seaborn as sns

from pandas.plotting import parallel_coordinates

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"

path = "../data/"

percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

In [4]:
df_study = pd.read_csv(rf'{export}data_complete.csv', sep= ";")
display(df_study)

,Unnamed: 0,DateTime,State,mappingMethod,TaskNo,TargetLayers,LayerCount,TrialIdx,posX,posY,posZ,TimeStamp,iteractionType,currentLayer,Proband
0,0,2021-06-29T07:13:59.622Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23
1,1,2021-06-29T07:13:59.630Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23
2,2,2021-06-29T07:13:59.673Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23
3,3,2021-06-29T07:13:59.707Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23
4,4,2021-06-29T07:13:59.742Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2733075,2733075,2021-05-31T10:19:26.341Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12
2733076,2733076,2021-05-31T10:19:26.416Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12
2733077,2733077,2021-05-31T10:19:26.444Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12
2733078,2733078,2021-05-31T10:19:26.474Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12


In [5]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly_removed-inactive-300.csv', sep= ";")
display(df_cleaned)

,DateTime,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,...,iteractionType,currentLayer,Proband,shifted,Result,Block,Target,Target_Relative,DateTime2,FrameTime
0,2021-06-29 07:19:27.465000+00:00,0,65,65,VIEW,direct,1,"['3', '11', '9', '6', '1']",12,0,...,NaN,NaN,23,1.0,START,1,3,0.250000,2021-06-29 07:19:27.465000+00:00,NaN
1,2021-06-29 07:19:27.498000+00:00,1,66,66,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.498000+00:00,33.0
2,2021-06-29 07:19:27.540000+00:00,2,67,67,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.540000+00:00,42.0
3,2021-06-29 07:19:27.598000+00:00,3,68,68,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.598000+00:00,58.0
4,2021-06-29 07:19:27.637000+00:00,4,69,69,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.637000+00:00,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398926,2021-05-31 10:18:49.509000+00:00,1535230,2360990,90024,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.509000+00:00,32.0
1398927,2021-05-31 10:18:49.544000+00:00,1535231,2360991,90025,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.544000+00:00,35.0
1398928,2021-05-31 10:18:49.571000+00:00,1535232,2360992,90026,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.571000+00:00,27.0
1398929,2021-05-31 10:18:49.602000+00:00,1535233,2360993,90027,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.602000+00:00,31.0


In [10]:
def computeCI(df, m= 'mean', c='count', s='std'):
    df['ci95_hi'] = df[m] + 1.96*df[s]/(df[c].apply(np.sqrt))
    df['ci95_lo'] = df[m] - 1.96*df[s]/(df[c].apply(np.sqrt))

def computeWhiskers(df, desc, col_names, q1='25%', q3='75%'):
    desc['iqr'] = desc[q3] - desc[q1]

    # Whisker-Berechnung (innerhalb 1.5 * IQR)
    desc['lower_bound'] = desc[q1] - 1.5 * desc['iqr']
    desc['upper_bound'] = desc[q3] + 1.5 * desc['iqr']    

    lower_whiskers = []
    upper_whiskers = []

    for col_name in col_names:
        lower_bound = desc['lower_bound'].T[col_name]
        upper_bound = desc['upper_bound'].T[col_name]

        # Whiskers sind die letzten Punkte innerhalb dieser Grenzen
        l = df[df[col_name] >= lower_bound][col_name].min()
        u = df[df[col_name] <= upper_bound][col_name].max()

        lower_whiskers.append(l)
        upper_whiskers.append(u)

    desc['lower_whisker'] = lower_whiskers
    desc['upper_whisker'] = upper_whiskers

In [12]:
df_study['FrameTime'] = pd.to_datetime(df_study["DateTime"], format=timeFormat).diff().dt.microseconds / 1000.0

In [16]:
desc_complete = pd.DataFrame(df_study['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_complete)

computeWhiskers(df_study, desc_complete, ['FrameTime'])

display(desc_complete.T)

desc_complete.T.to_csv(rf'{export}desc-stats_framestats_complete.csv', sep=';')


,FrameTime
count,2.733079e+06
mean,3.750637e+01
std,8.842094e+00
min,0.000000e+00
5%,2.700000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.900000e+01


In [ ]:
desc_interaction = pd.DataFrame(df_cleaned['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_interaction)

computeWhiskers(df_cleaned, desc_interaction, ['FrameTime'])

display(desc_interaction.T)

desc_interaction.T.to_csv(rf'{export}desc-stats_framestats_interaction.csv', sep=';')

,FrameTime
count,1.398930e+06
mean,3.966261e+01
std,3.723145e+01
min,0.000000e+00
5%,2.700000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.900000e+01
